In [ ]:
%%sql -r use_role
USE ROLE SYSADMIN;

In [ ]:
%%sql -r use_wh
USE WAREHOUSE COMPUTE_WH;

In [ ]:
# データ準備の確認：setup/setup.sqlが正しく完了しているか確認
from snowflake.snowpark.context import get_active_session
session = get_active_session()

try:
    chk = session.table("DF_HANDSON.CONTRACT.CHECK_RESULTS").to_pandas()
    session.table("DF_HANDSON.ML.DAILY").limit(1).collect()
except Exception as e:
    raise RuntimeError("データがまだありません。先に setup/setup.sql をワークシートで実行（Run All）してください") from e

errors = chk[(~chk["PASSED"]) & (chk["SEVERITY"] == "ERROR")]
if len(errors):
    raise RuntimeError("データの契約チェックでエラーがあります\n" + errors.to_string(index=False))
warns = chk[(~chk["PASSED"]) & (chk["SEVERITY"] == "WARN")]
if len(warns):
    print("注意:\n" + warns.to_string(index=False))
print("データは準備できています")

In [ ]:
# 地図の準備：地図の描画に使用する関数を用意しておきます
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

FMT_JA = {"STATION_IN": "駅ナカ", "STATION_FRONT": "駅前", "RESIDENTIAL": "住宅街"}
FMT_COLOR = {"駅ナカ": "#1f77b4", "駅前": "#ff7f0e", "住宅街": "#2ca02c", "不明": "#7f7f7f"}

_wards = session.table("DF_HANDSON.REF.WARD").to_pandas()
WARD_FC = {"type": "FeatureCollection",
           "features": [{"type": "Feature", "id": r.WARD_CODE, "properties": {"ward_code": r.WARD_CODE, "name": r.WARD_NAME},
                         "geometry": json.loads(r.GEOJSON)} for r in _wards.itertuples()]}
RAIL_FC = {"type": "FeatureCollection",
           "features": [{"type": "Feature", "properties": {}, "geometry": json.loads(g)}
                        for g in session.table("DF_HANDSON.REF.RAIL").to_pandas()["GEOJSON"]]}
_sea = {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {}, "geometry":
        {"type": "Polygon", "coordinates": [[[139.4, 35.4], [140.1, 35.4], [140.1, 35.95], [139.4, 35.95], [139.4, 35.4]]]}}]}
BASE_LAYERS = [
    {"source": _sea, "type": "fill", "color": "#e8eef2", "below": "traces"},
    {"source": WARD_FC, "type": "fill", "color": "#f7f5f0", "below": "traces"},
    {"source": WARD_FC, "type": "line", "color": "#9a9a9a", "line": {"width": 0.8}, "below": "traces"},
    {"source": RAIL_FC, "type": "line", "color": "#b0b0b0", "line": {"width": 1.0}, "below": "traces"},
]
MAP_LAYOUT = dict(map_layers=BASE_LAYERS, margin={"l": 0, "r": 0, "t": 40, "b": 0})
CENTER = {"lat": 35.69, "lon": 139.74}

def fmt_ja(s):
    return s.map(FMT_JA).fillna("不明")

def show_map(df, title="", zoom=10.2, **kw):
    """緯度(LAT)・経度(LON)の列を持つ表を、23区の地図に点で表示します"""
    fig = px.scatter_map(df, lat="LAT", lon="LON", map_style="white-bg", center=CENTER, zoom=zoom,
                         height=600, title=title, **kw)
    fig.update_layout(**MAP_LAYOUT)
    fig.show()

## 1. 取り込んだデータを確認する

In [ ]:
%%sql -r stores
SELECT s.store_id, s.store_name, s.store_format, s.lat, s.lon, s.open_date,
       ROUND(AVG(d.sales)) AS avg_daily_sales
FROM DF_HANDSON.CONTRACT.STORE s
CROSS JOIN DF_HANDSON.ML.CONFIG c
LEFT JOIN DF_HANDSON.ML.DAILY d ON d.store_id = s.store_id AND d.ds > DATEADD(day, -28, c.data_end) AND d.sales IS NOT NULL
GROUP BY ALL
ORDER BY s.store_id;

In [ ]:
df = stores.to_pandas()
df["業態"] = fmt_ja(df["STORE_FORMAT"])
df["SIZE"] = df["AVG_DAILY_SALES"].fillna(df["AVG_DAILY_SALES"].min())   # 開業予定の店は売上がないので最小サイズで表示
print("店舗数：", len(df), "（うち開業予定：", int(df["AVG_DAILY_SALES"].isna().sum()), "）")
show_map(df, title="SNOW BAKERY のお店（円の大きさ＝直近4週間の平均日販）", color="業態", size="SIZE", size_max=18,
         color_discrete_map=FMT_COLOR, hover_name="STORE_NAME", hover_data={"AVG_DAILY_SALES": ":,.0f", "OPEN_DATE": True, "SIZE": False})

In [ ]:
-- 業態ごとに、曜日別の平均売上を「その業態の平均＝100」で表します（祝日・年末年始・お盆を除く）
WITH x AS (
  SELECT store_format, DAYOFWEEKISO(ds) AS dow, AVG(sales) AS avg_sales
  FROM DF_HANDSON.ML.DAILY
  WHERE sales IS NOT NULL AND NOT is_holiday AND NOT is_special_period
  GROUP BY 1, 2)
SELECT store_format, dow, DECODE(dow, 1, '月', 2, '火', 3, '水', 4, '木', 5, '金', 6, '土', 7, '日') AS weekday,
       ROUND(100 * avg_sales / AVG(avg_sales) OVER (PARTITION BY store_format)) AS index_100
FROM x 
ORDER BY store_format, dow;

In [ ]:
d = dow_pattern.to_pandas()
d["業態"] = fmt_ja(d["STORE_FORMAT"])
fig = px.bar(d, x="WEEKDAY", y="INDEX_100", color="業態", barmode="group", color_discrete_map=FMT_COLOR,
             title="曜日ごとの売上（各業態の平均＝100）", labels={"WEEKDAY": "", "INDEX_100": "指数"}, height=380)
fig.add_hline(y=100, line_dash="dot", line_color="grey")
fig.show()

駅ナカの店は**平日**に、住宅街の店は**週末**に売れてそうだということがわかりました。同じチェーンでも、店の場所によって売れ方の傾向が違います。  
機械学習を使うことでこの傾向の差を人が一つずつ調べなくても、売上を予測することができます。

## 2. 来月の売上を店ごとに予測する
業態と売上規模の違う **12店** を選び、先月までの毎日の売上から **今月（テスト月）の毎日の売上** を予測していきます。

In [ ]:
%%sql -r fc_targets
SELECT t.store_id, s.store_name, t.store_format, t.avg_sales
FROM DF_HANDSON.ML.FC_TARGET_STORES t JOIN DF_HANDSON.CONTRACT.STORE s ON s.store_id = t.store_id
ORDER BY t.store_format, t.avg_sales DESC;

In [ ]:
SELECT * FROM DF_HANDSON.ML.FC_TRAIN ORDER BY ts DESC, store_id LIMIT 10;

## 特徴量の説明

### ML関数 **FORECAST**
時系列データの予測モデルをSQLのみで作成することができます。  
```sql
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST
```

In [ ]:
CREATE OR REPLACE [穴埋め] DF_HANDSON.ML.SALES_FORECAST(
  INPUT_DATA        => SYSTEM$REFERENCE('VIEW', 'DF_HANDSON.ML.FC_TRAIN'),
  SERIES_COLNAME    => 'STORE_ID',     -- 店ごとに別々に予測
  TIMESTAMP_COLNAME => 'TS',           -- 日付
  TARGET_COLNAME    => 'SALES',        -- 予測したい値（残りの列は外生変数として自動で使われる）
  CONFIG_OBJECT     => {'evaluate': FALSE});

予測をするときもSQLで実行できます。
```sql
DF_HANDSON.ML.SALES_FORECAST!FORECAST
```

In [ ]:
CREATE OR REPLACE TABLE DF_HANDSON.ML.FC_RESULT AS
SELECT series::STRING AS store_id, ts::DATE AS ds, forecast, lower_bound, upper_bound
FROM TABLE(DF_HANDSON.ML.SALES_FORECAST[穴埋め](
  INPUT_DATA        => SYSTEM$REFERENCE('VIEW', 'DF_HANDSON.ML.FC_FUTURE'),
  SERIES_COLNAME    => 'STORE_ID',
  TIMESTAMP_COLNAME => 'TS'));

In [ ]:
%%sql -r fc_score
-- WAPE ＝ 毎日の誤差（の絶対値）の合計 ÷ 実績の合計。「1か月でどれくらい外れたか」を％で表します
SELECT r.store_id, s.store_name, s.store_format,
       ROUND(SUM(d.sales)) AS actual, ROUND(SUM(r.forecast)) AS forecast,
       ROUND(SUM(ABS(r.forecast - d.sales)) / NULLIF(SUM(d.sales), 0) * 100, 1) AS wape_pct
FROM DF_HANDSON.ML.FC_RESULT r
JOIN DF_HANDSON.ML.DAILY d ON d.store_id = r.store_id AND d.ds = r.ds
JOIN DF_HANDSON.CONTRACT.STORE s ON s.store_id = r.store_id
GROUP BY ALL ORDER BY wape_pct;

In [ ]:
%%sql -r fc_chart
SELECT d.store_id, s.store_name, d.ds, d.sales AS actual, r.forecast, r.lower_bound, r.upper_bound
FROM DF_HANDSON.ML.DAILY d
JOIN DF_HANDSON.ML.FC_TARGET_STORES t ON t.store_id = d.store_id
JOIN DF_HANDSON.CONTRACT.STORE s ON s.store_id = d.store_id
CROSS JOIN DF_HANDSON.ML.CONFIG c
LEFT JOIN DF_HANDSON.ML.FC_RESULT r ON r.store_id = d.store_id AND r.ds = d.ds
WHERE d.ds BETWEEN DATEADD(day, -42, c.test_start) AND c.test_end
ORDER BY d.store_id, d.ds;

In [ ]:
d = fc_chart.to_pandas()
names = list(dict.fromkeys(d["STORE_NAME"]))
fig = go.Figure()
for i, n in enumerate(names):
    x = d[d["STORE_NAME"] == n]
    v = i == 0
    fig.add_trace(go.Scatter(x=x["DS"], y=x["UPPER_BOUND"], line={"width": 0}, showlegend=False, visible=v, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=x["DS"], y=x["LOWER_BOUND"], fill="tonexty", line={"width": 0}, name="予測の幅", visible=v,
                             fillcolor="rgba(255,127,14,0.2)"))
    fig.add_trace(go.Scatter(x=x["DS"], y=x["ACTUAL"], name="実績", line={"color": "black"}, visible=v))
    fig.add_trace(go.Scatter(x=x["DS"], y=x["FORECAST"], name="予測", line={"color": "#ff7f0e"}, visible=v))
buttons = [dict(label=n, method="update",
                args=[{"visible": [j // 4 == i for j in range(4 * len(names))]}, {"title": f"{n}：毎日の売上（実績と予測）"}])
           for i, n in enumerate(names)]
fig.update_layout(updatemenus=[dict(buttons=buttons, x=0, y=1.18, xanchor="left")], height=420,
                  title=f"{names[0]}：毎日の売上（実績と予測）", yaxis_title="円")
fig.show()

In [ ]:
%%sql -r fc_fi
CALL DF_HANDSON.ML.SALES_FORECAST!EXPLAIN_FEATURE_IMPORTANCE();

In [ ]:
fi = fc_fi.to_pandas()
fi.columns = [c.upper() for c in fi.columns]
score_col = "SCORE" if "SCORE" in fi.columns else [c for c in fi.columns if c not in ("SERIES", "RANK", "FEATURE_NAME", "FEATURE_TYPE")][0]
JA = {"is_promo": "販促", "is_holiday": "祝日", "is_special": "年末年始・お盆", "temp_avg": "気温", "precip_mm": "降水量"}
fi["要因"] = fi["FEATURE_NAME"].map(lambda f: JA.get(str(f).lower(), str(f)))
imp = fi.groupby("要因")[score_col].mean().sort_values().tail(10)
px.bar(imp, orientation="h", title="12店で平均した特徴量の重要度（上位10）", labels={"value": "重要度", "要因": ""},
       height=380).update_layout(showlegend=False).show()
print("※ 日付や過去の売上から自動で作られた特徴量は英語名のまま表示しています")